Document Loading (PyPDF Loader)

In [ ]:
! pip install -qU langchain-community pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "/content/attention_is_all_you_need.pdf"
loader = PyPDFLoader(file_path)
docs = loader.load()
print(docs)
print(docs[0].metadata)
print(docs[0].page_content)

Text Splitting (Recursive Character-Text Splitter)

In [ ]:
!pip install -qU langchain-text-splitter

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)
print(all_splits)

print(f"Split blog post into {len(all_splits)} sub-documents.")
print(f"Metadata: {all_splits[0].metadata}")

Embeddings (Hugging Face-Embeddings)

In [ ]:
! pip install -qU  langchain langchain-huggingface sentence_transformers

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize free, local embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

Vector Store (Chroma DB)

In [ ]:
pip install -qU langchain-chroma

In [ ]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)
document_ids = vector_store.add_documents(documents=all_splits)
sample = vector_store.get(limit=1, include=["embeddings", "documents"])
print(f"Embedding dimensions: {len(sample['embeddings'][0])}")
print(sample)
print(document_ids[:3])

LLM (Gemini Model)

In [ ]:
!pip install -qU langchain-google-genai

In [ ]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

In [ ]:
api_key=userdata.get('gemini_api_key')

In [ ]:
model = init_chat_model(
    "google_genai:gemini-2.5-flash",
    api_key=api_key,

)

Convert Retrieval Function to a Tool

In [ ]:
from langchain.tools import tool

In [ ]:
@tool
def retrieve_from_pdf(query: str, k: int = 2)-> str:
    """Retrieve information from the Attention Is All You Need research paper."""

    retrieved_docs = vector_store.similarity_search(query, k=k)

    # Build context string
    docs_content = ""
    for doc in retrieved_docs:
        docs_content += f"Source: {doc.metadata}\n"
        docs_content += f"Content: {doc.page_content}\n\n"

    return docs_content, retrieved_docs

Add Web Search Tool

In [ ]:
!pip install langchain langchain-tavily

In [ ]:
from langchain_tavily import TavilySearch

tavily_api_key = userdata.get("tavily_apikey")

web_search_tool = TavilySearch(
    max_results = 5,
    search_depth = "advanced",
    tavily_api_key = tavily_api_key,
)

In [ ]:
system_prompt="""You are a helpful research assistant with access to two tools:

1. retrieve_from_pdf: Use this to find information from the
   "Attention Is All You Need" research paper

2. TavilySearch: Use this to find current information
   not in the paper (recent events, updates, etc.)

Strategy:
- For questions about the paper content → use retrieve_from_pdf
- For questions about recent events or topics not in the paper → use TavilySearch
- DON'T make up things
"""

Create and Configure the Agent

In [ ]:
from langchain.agents import create_agent
agent = create_agent(
    model=model,
    tools=[retrieve_from_pdf, web_search_tool],
    system_prompt=system_prompt,
    debug=True,
)

Execute the Agent

In [ ]:
user_query = """Compare the attention mechanism from the paper with recent improvements like Flash Attention, and tell me which approach would be better for my college project."""

response = agent.invoke({
    "messages": [{"role": "user", "content": user_query}]
})

print(response["messages"][-1].content[0]["text"])